# Using The Decay Heat Subpackage

In this tutorial we'll show an example of constructing decay heat profiles.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rc

rc("font", family="serif", size=15)
rc("savefig", dpi=600)
rc("figure", figsize=(8, 6))
rc("mathtext", fontset="dejavuserif")
rc("lines", linewidth=3)

In [ ]:
from stream.physical_models.decay_heat import (
    actinides,
    activation,
    fission_products,
    fissions,
)
from stream.units import MeV
from stream.utilities import factor

Q: MeV = 201.7

## Using Fission Products


### Comparing Fission Product Profiles

Let us explore the available standards for fission product decay heat profiles:

In [ ]:
from stream.physical_models.decay_heat.fission_products import Source, Standard

In [ ]:
from itertools import product

from more_itertools import filter_except

possible_entries = product(Standard, Source)
existing_entries = list(filter_except(lambda x: fission_products.contribution(*x), possible_entries, NameError))

time = np.logspace(-1, 3)


def plot_fp(norm: bool):

    for st, so in existing_entries:
        f = fission_products.contribution(st, so)
        plt.semilogx(time, 100 * f(time) / (f(0) if norm else Q), label=f"{st.name}, {so.name}")
    plt.grid()
    plt.xlabel("Time [s]")


plt.figure(figsize=(16, 12))
ax1 = plt.subplot(221)
plot_fp(norm=True)
plt.ylabel("Profile [%]")

ax2 = plt.subplot(222)
plot_fp(norm=False)
plt.ylabel("Contribution [% of Power]")
plt.legend();

In [ ]:
fp = fission_products.contribution(Standard.ANS73, Source.U235)

## Using Actinides

In [ ]:
time = np.logspace(0, 7)
R = 1
U239 = actinides.U239_profile(time, np.inf)
Np239 = actinides.Np239_profile(time, np.inf)
ac = actinides.contribution(R=R)


plt.figure(figsize=(16, 12))

ax1 = plt.subplot(2, 2, 1)
plt.plot(time, 100 * U239)
plt.plot(time, 100 * Np239)
plt.ylabel("Profile [%]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.grid()

ax2 = plt.subplot(2, 2, 2)
plt.plot(time, 100 * U239 * R * actinides.E_U / Q, label=r"$^{239}U$")
plt.plot(time, 100 * Np239 * R * actinides.E_Np / Q, label=r"$^{239}Np$")
plt.plot(time, 100 * ac(time, np.inf) / Q, label=r"$^{239}U + ^{239}Np$")
plt.ylabel("Contribution [% of Power]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.legend()
plt.grid()

## Using Activation

In [ ]:
plt.figure(figsize=(16, 12))

Al28_profile = activation.profile(5.1600e-03)
al = factor(Al28_profile, by=3.02 * 0.6)

ax1 = plt.subplot(2, 2, 1)
plt.plot(time, 100 * Al28_profile(time, np.inf))
plt.ylabel("Profile [%]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.grid()

ax2 = plt.subplot(2, 2, 2)
plt.plot(time, 100 * al(time, np.inf) / Q, label=r"$^{28}Al$")
plt.ylabel("Contribution [% of Power]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.legend()
plt.grid()

## Using (Delayed) Fissions

In [ ]:
from stream.calculations.point_kinetics import ReactivityController
from stream.units import pcm
from stream.utilities import factor, normalize

time = np.logspace(-2, 2)
dollar = 727.5 * pcm

f_profile = fissions.profile(
    time=np.logspace(-5, 5, 1000),
    generation_time=43.74e-6,
    delayed_neutron_fractions=dollar * normalize([27.3, 74.75, 256.75, 127.4, 142.35, 21.45]),
    delayed_groups_decay_rates=np.array([3.01, 1.14, 0.301, 0.111, 0.0305, 0.0124]),
    controls=ReactivityController(lambda _, __, t: (-2e3 * t if t < 5 else -10e3) * pcm),
)
Q_prompt = float(Q - fp(0) - al(0) - ac(0))
f = factor(f_profile, Q_prompt)

plt.figure()
plt.plot(time, 100 * f_profile(time))
plt.ylabel("Profile [%]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.grid()

## Adding Everything Together

In [ ]:
time = np.logspace(-2, 5, 1000)


def total(t, T=np.inf):
    return sum(f(t, T) for f in (f, fp, al, ac))


plt.figure()
plt.plot(time, 100 * f(time) / Q, label="Fissions")
plt.plot(time, 100 * fp(time) / Q, label="Fission Products")
plt.plot(time, 100 * al(time) / Q, label="Al Activation")
plt.plot(time, 100 * ac(time) / Q, label="Actinides")
plt.plot(time, 100 * total(time) / Q, label="Total", color="k")
plt.ylabel("Contribution [% of Power]")
plt.xlabel("Time [s]")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.ylim(1e-4, 1.3e2)
plt.grid()